# 19 — Robustness Benchmarking & Red-Teaming Toolkits — Lab

## Description

This lab exercises automated vulnerability auditing using IBM ART. You load a pretrained CIFAR-10 classifier, run three standardized attacks (FGSM, PGD, C&W), record per-attack metrics (fooling rate and perturbation magnitudes), and produce a structured vulnerability report. You then situate the model on the RobustBench leaderboard to contextualize its robustness.

## Objectives

- Student can use IBM ART to run a multi-attack robustness audit and produce a structured vulnerability report with per-attack metrics.
- Student can explain what AutoAttack is, why it uses four complementary attacks, and why it is preferred over single-attack evaluation for leaderboard comparisons.
- Student can situate a model on the RobustBench leaderboard by its AutoAttack accuracy.

## Prerequisites

You should have completed **Lesson 06** (AI Development Lifecycle and MLOps) to understand model versioning and experiment tracking. You should have studied m13 (Physical-World Attacks and Universal Perturbations) for familiarity with perturbation budgets and attack evaluation. You must have completed the lecture content for this module (**Lesson 19**) covering **RobustBench**, **AutoAttack**, ART architecture, and red-teaming phases.

## Required Software/Packages

- PyTorch >= 1.9.0
- adversarial-robustness-toolbox >= 1.10.0
- numpy >= 1.19.0
- pandas >= 1.1.0
- torchvision >= 0.10.0 (for CIFAR-10 dataset)

## Environment/Setup Instructions

Run the cell below to import all required libraries and verify that ART and PyTorch are installed correctly.

In [31]:
# If you want to load models from RobustBench, you need to install it.
import warnings
warnings.filterwarnings('ignore')


# Art
try:
    import art
    print("ART already installed.")
except ImportError:
    !pip install -q adversarial-robustness-toolbox
    import art
    print("ART installed successfully.")

# RobustBench
try:
    import robustbench
except ImportError:
    !pip install -q git+https://github.com/RobustBench/robustbench.git
    import robustbench
    print("RobustBench installed successfully.")

ART already installed.


Let's install [AutoAttack](https://arxiv.org/abs/2003.01690) from
ICML 2020 with 2/4 attacks (only APGD-CE and APGD-DLR)

In [32]:
# now let's get the ## AutoAttack evaluation
try:
    import autoattack
    print("AutoAttack already installed.")
except ImportError:
    !pip install -q git+https://github.com/fra31/auto-attack
    import autoattack
    print("AutoAttack installed successfully.")

AutoAttack already installed.


In [33]:

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, CarliniL2Method
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("All imports successful. Ready to begin.")

PyTorch version: 2.14.1
CUDA available: False
All imports successful. Ready to begin.


## Background/Context

In the lecture, you learned that AutoAttack is an ensemble of four complementary attacks, and that RobustBench uses AutoAttack to provide standardized robustness evaluation. In practice, organizations run multi-attack audits using tools like IBM ART to assess model vulnerability before deployment. This lab replicates the core audit workflow: you load a model, run three key attacks from the AutoAttack ensemble (APGD-CE approximated by PGD, APGD-DLR approximated by C&W, and a fast boundary attack approximated by C&W), and produce a vulnerability report that summarizes the findings. The report shows per-attack fooling rate (percentage of examples fooled) and perturbation magnitudes, allowing you to prioritize defenses based on which attack is most effective.

## Exercise Instructions

1. Define a simple CNN model for CIFAR-10 classification with two convolutional layers and a fully connected head. (2 min)
2. Load the CIFAR-10 test dataset, selecting the first 200 examples for this audit. (2 min)
3. Create a PyTorchClassifier wrapper by passing your model and preprocessing function to ART. (3 min)
4. Instantiate three attacks via ART: FGSM (epsilon=8/255), PGD-20 (epsilon=8/255, alpha=2/255, max_iter=20), and C&W L2 (kappa=0, max_iter=50). (5 min)
5. Run each attack on the test batch and record: fooling rate (%), mean L2 perturbation magnitude, and mean L-infinity perturbation magnitude. (10 min)
6. Create a structured audit report as a pandas DataFrame with one row per attack and the three metrics as columns. (5 min)
7. Compare your model's implied AutoAttack accuracy to the RobustBench leaderboard snapshot provided. (3 min)

### Define model and create ART classifier

**Options**

- Train your own model
- Load pre-trained model

### Load the data

In [34]:

"""
# Load CIFAR-10 test data
transform = transforms.Compose([transforms.ToTensor()])
cifar10_test = datasets.CIFAR10(root='/tmp/cifar10', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(cifar10_test, batch_size=200, shuffle=False)

x_test, y_test = next(iter(test_loader))
x_test = x_test.numpy()
y_test = y_test.numpy()
print(f"Test batch shape: {x_test.shape}, Test labels shape: {y_test.shape}")

"""

# Download subset of CIFAR-10 datseta from torch vision

import os
from torch.utils.data import DataLoader, Subset, TensorDataset
import urllib.request

os.makedirs("cifar10_subset", exist_ok=True)


base_url = "https://raw.githubusercontent.com/Abuhamad/comp378_f26/main/datasets/cifar10_subset/"
for fname in ["train.pt", "test.pt"]:
    path = os.path.join("cifar10_subset", fname)
    if not os.path.exists(path):
        urllib.request.urlretrieve(base_url + fname, path)
        print(f"Downloaded {fname}")

# Load datasets
train_data = torch.load("cifar10_subset/train.pt", weights_only=True) # contains 5000 train images of cifar10 dataset - train
test_data = torch.load("cifar10_subset/test.pt", weights_only=True) # contains 1000 test images of cifar10 dataset - test

train_dataset = TensorDataset(train_data["images"].permute(0, 3, 1, 2).float() / 255.0, train_data["labels"])
test_dataset = TensorDataset(test_data["images"].permute(0, 3, 1, 2).float() / 255.0, test_data["labels"])

# Subsets for quick training and evaluation
trainset_subset = Subset(train_dataset, range(500))
testset_subset = Subset(test_dataset, range(100))

train_loader = DataLoader(trainset_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset_subset, batch_size=50, shuffle=False)


In [35]:
# Option 1: Define your own model

"""
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x


model = SimpleCNN().to(device)
print(f"Model initialized on {device}")

"""

'\nclass SimpleCNN(nn.Module):\n    def __init__(self):\n        super(SimpleCNN, self).__init__()\n        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)\n        self.relu1 = nn.ReLU()\n        self.pool1 = nn.MaxPool2d(2, 2)\n        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)\n        self.relu2 = nn.ReLU()\n        self.pool2 = nn.MaxPool2d(2, 2)\n        self.fc1 = nn.Linear(64 * 8 * 8, 128)\n        self.relu3 = nn.ReLU()\n        self.fc2 = nn.Linear(128, 10)\n\n    def forward(self, x):\n        x = self.pool1(self.relu1(self.conv1(x)))\n        x = self.pool2(self.relu2(self.conv2(x)))\n        x = x.view(x.size(0), -1)\n        x = self.relu3(self.fc1(x))\n        x = self.fc2(x)\n        return x\n\n\nmodel = SimpleCNN().to(device)\nprint(f"Model initialized on {device}")\n\n'

In [36]:
# Option 2: Load a pre-trained model
# Example using torchvision or RobustBench

from robustbench.utils import load_model
from robustbench.model_zoo.models import model_dicts

print("Top-level keys in RobustBench model_dicts:", list(model_dicts.keys()))

# Safely inspect the nested dataset structure
for dataset_key in model_dicts.keys():
    print(f"\nDataset: {dataset_key}")
    for threat_key in model_dicts[dataset_key].keys():
        models_list = list(model_dicts[dataset_key][threat_key].keys())
        print(f"  Threat Model: {threat_key} -> Found {len(models_list)} models.")
        print(f"  Examples: {models_list[:5]}")

Top-level keys in RobustBench model_dicts: [<BenchmarkDataset.cifar_10: 'cifar10'>, <BenchmarkDataset.cifar_100: 'cifar100'>, <BenchmarkDataset.imagenet: 'imagenet'>]

Dataset: BenchmarkDataset.cifar_10
  Threat Model: ThreatModel.Linf -> Found 76 models.
  Examples: ['Andriushchenko2020Understanding', 'Carmon2019Unlabeled', 'Sehwag2020Hydra', 'Wang2020Improving', 'Hendrycks2019Using']
  Threat Model: ThreatModel.L2 -> Found 21 models.
  Examples: ['Augustin2020Adversarial', 'Engstrom2019Robustness', 'Rice2020Overfitting', 'Rony2019Decoupling', 'Standard']
  Threat Model: ThreatModel.corruptions -> Found 17 models.
  Examples: ['Diffenderfer2021Winning_LRR', 'Diffenderfer2021Winning_LRR_CARD_Deck', 'Diffenderfer2021Winning_Binary', 'Diffenderfer2021Winning_Binary_CARD_Deck', 'Rebuffi2021Fixing_70_16_cutmix_extra_Linf']

Dataset: BenchmarkDataset.cifar_100
  Threat Model: ThreatModel.Linf -> Found 38 models.
  Examples: ['Gowal2020Uncovering', 'Gowal2020Uncovering_extra', 'Cui2020Learna

In [37]:
# load your choice ofpre-trained model from RobustBench
model = load_model(
    model_name="Carmon2019Unlabeled",  # model from the CIFAR-10 leaderboard
    dataset="cifar10",
    threat_model="Linf"   # "Linf", "L2", or "corruptions"
)
model.eval()  # don't forget eval mode
model = model.to(device)  # move model to the appropriate device

In [38]:
# Extract x_test and y_test from test_loader
# Iterate through the loader to gather 200 examples as specified in the instructions
x_test_list, y_test_list = [], []
for batch_x, batch_y in test_loader:
    x_test_list.append(batch_x.numpy())
    y_test_list.append(batch_y.numpy())
    if sum(len(b) for b in x_test_list) >= 200: # stop when we reach 200 examples
        break

x_test = np.concatenate(x_test_list, axis=0)[:200]
y_test = np.concatenate(y_test_list, axis=0)[:200]

# Create ART PyTorchClassifier
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

classifier = PyTorchClassifier(
    model=model,
    loss=loss_fn,
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=10,
    device_type="gpu" if torch.cuda.is_available() else "cpu"
)

predictions = classifier.predict(x_test)
clean_accuracy = np.mean(np.argmax(predictions, axis=1) == y_test)
print(f"Clean accuracy on test examples: {clean_accuracy:.1%}")

Clean accuracy on test examples: 89.0%


### Auditing using ART

In [40]:
# Steps 4-6: Run attacks using ART and create audit report
epsilon = 8/255
attack_results = {}

def evaluate_attack(classifier, x_adv, x_clean, y_true, attack_name, results_dict):
    """
    Compute fooling rate and perturbation norms for an adversarial attack,
    store them in results_dict, and print a summary.

    Returns the metrics dict.
    """
    pred_adv = classifier.predict(x_adv)
    fooling_rate = np.mean(np.argmax(pred_adv, axis=1) != y_true) * 100

    pert = x_adv - x_clean
    mean_l2 = np.mean(np.linalg.norm(pert.reshape(pert.shape[0], -1), axis=1))
    mean_linf = np.mean(np.abs(pert).max(axis=(1, 2, 3)))

    metrics = {'fooling_rate': fooling_rate, 'mean_l2': mean_l2, 'mean_linf': mean_linf}
    results_dict[attack_name] = metrics

    print(f"{attack_name} - Fooling rate: {fooling_rate:.1f}%, "
          f"Mean L2: {mean_l2:.4f}, Mean Linf: {mean_linf:.4f}")
    return metrics

# FGSM attack
print("Running FGSM attack...")
fgsm = FastGradientMethod(estimator=classifier, eps=epsilon)
x_adv_fgsm = fgsm.generate(x=x_test)
attack_results['FGSM'] = evaluate_attack(classifier,
                                         x_adv_fgsm,
                                         x_test, y_test,
                                         'FGSM',
                                         attack_results)

# PGD-20 attack
print("\nRunning PGD-20 attack...")
pgd = ProjectedGradientDescent(estimator=classifier,
                               eps=epsilon,
                               eps_step=2/255,
                               max_iter=20,
                               num_random_init=1)
x_adv_pgd = pgd.generate(x=x_test)
attack_results['PGD-20'] = evaluate_attack(classifier,
                                           x_adv_pgd,
                                           x_test, y_test,
                                           'PGD-20',
                                           attack_results)

# C&W L2 attack
print("\nRunning C&W L2 attack...")
# Fixed: Changed estimator=classifier to classifier=classifier
cw = CarliniL2Method(classifier=classifier,
                     max_iter=10, # you can increase it to 50
                     confidence=0.0)
x_adv_cw = cw.generate(x=x_test)
attack_results['C&W L2'] = evaluate_attack(classifier,
                                           x_adv_cw,
                                           x_test,
                                           y_test,
                                           'C&W L2',
                                           attack_results)

# Create structured audit report
audit_report = pd.DataFrame.from_dict(attack_results, orient='index')
audit_report.columns = ['Fooling Rate (%)', 'Mean L2 Magnitude', 'Mean L∞ Magnitude']
audit_report = audit_report.round(4)

print("\n" + "="*70)
print("VULNERABILITY AUDIT REPORT")
print("="*70)
print(f"Model: Carmon2019Unlabeled on CIFAR-10")
print(f"Test set size: {len(x_test)} examples")
print(f"Clean accuracy: {clean_accuracy:.1%}")
print("\nPer-Attack Metrics:")
print(audit_report)

# Determine primary defense recommendation
max_fooling_attack = audit_report['Fooling Rate (%)'].idxmax()
max_fooling_rate = audit_report['Fooling Rate (%)'].max()

print("\nDefense Recommendation:")
if max_fooling_rate > 80:
    severity = "CRITICAL"
elif max_fooling_rate > 50:
    severity = "HIGH"
elif max_fooling_rate > 20:
    severity = "MEDIUM"
else:
    severity = "LOW"

print(f"Severity: {severity}")
print(f"Primary attack: {max_fooling_attack} ({max_fooling_rate:.1f}% fooling rate)")
print(f"Recommended action: {'Adversarial training' if max_fooling_rate > 50 else 'Input preprocessing or ensemble defenses'}")
print("="*70)

Running FGSM attack...
FGSM - Fooling rate: 24.0%, Mean L2: 1.7388, Mean Linf: 0.0314

Running PGD-20 attack...


PGD-20 - Fooling rate: 36.0%, Mean L2: 1.6689, Mean Linf: 0.0314

Running C&W L2 attack...


C&W L_2: 100%|██████████| 100/100 [32:01<00:00, 19.21s/it]


C&W L2 - Fooling rate: 89.0%, Mean L2: 0.9248, Mean Linf: 0.2407

VULNERABILITY AUDIT REPORT
Model: Carmon2019Unlabeled on CIFAR-10
Test set size: 100 examples
Clean accuracy: 89.0%

Per-Attack Metrics:
        Fooling Rate (%)  Mean L2 Magnitude  Mean L∞ Magnitude
FGSM                24.0             1.7388             0.0314
PGD-20              36.0             1.6689             0.0314
C&W L2              89.0             0.9248             0.2407

Defense Recommendation:
Severity: CRITICAL
Primary attack: C&W L2 (89.0% fooling rate)
Recommended action: Adversarial training


### Auditing using AutoAttack

In [41]:
# Run AutoAttack
from autoattack import AutoAttack

print("Running AutoAttack...")

# AutoAttack expects PyTorch tensors, so we convert our NumPy arrays
x_test_tensor = torch.from_numpy(x_test).to(device)
y_test_tensor = torch.from_numpy(y_test).to(device)

adversary = AutoAttack(model,
                       norm='Linf',
                       eps=8/255,
                       version='custom',
                       attacks_to_run=['apgd-ce', 'apgd-dlr'],
                       device=device)
adversary.apgd.n_restarts = 1

# Run evaluation on the tensors
x_adv = adversary.run_standard_evaluation(x_test_tensor, y_test_tensor)

Running AutoAttack...
using custom version including apgd-ce, apgd-dlr.
initial accuracy: 89.00%
apgd-ce - 1/1 - 31 out of 89 successfully perturbed
robust accuracy after APGD-CE: 58.00% (total time 334.4 s)
apgd-dlr - 1/1 - 5 out of 58 successfully perturbed
robust accuracy after APGD-DLR: 53.00% (total time 565.9 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 53.00%


In [42]:
import torch
import numpy as np
import pandas as pd
from autoattack import AutoAttack

# Initialize lists to store metrics for AutoAttack algorithms
aa_report_data = {}

# Helper function to compute and store metrics
def evaluate_adversarial_images(x_clean, x_perturbed, y_true, model, attack_name):
    # Ensure inputs are tensors
    if isinstance(x_perturbed, np.ndarray):
        x_perturbed_tensor = torch.from_numpy(x_perturbed).to(device)
    else:
        x_perturbed_tensor = x_perturbed.to(device)

    with torch.no_grad():
        outputs = model(x_perturbed_tensor)
        preds = outputs.argmax(dim=1).cpu().numpy()

    # Calculate Fooling Rate
    fooling_rate = np.mean(preds != y_true) * 100

    # Convert perturbed to numpy for distance calculation
    if not isinstance(x_perturbed, np.ndarray):
        x_p_np = x_perturbed.cpu().numpy()
    else:
        x_p_np = x_perturbed

    perturbation = x_p_np - x_clean
    mean_l2 = np.mean(np.linalg.norm(perturbation.reshape(perturbation.shape[0], -1), axis=1))
    mean_linf = np.mean(np.abs(perturbation).max(axis=(1, 2, 3)))

    return {
        'Fooling Rate (%)': fooling_rate,
        'Mean L2 Magnitude': mean_l2,
        'Mean L∞ Magnitude': mean_linf
    }

# 1. Run AutoAttack with individual components to isolate their outputs
x_sub, y_sub = x_test[:50], y_test[:50]
x_test_tensor = torch.from_numpy(x_sub).to(device)
y_test_tensor = torch.from_numpy(y_sub).to(device)

# Run APGD-CE
print("Evaluating APGD-CE from AutoAttack...")
aa_ce = AutoAttack(model, norm='Linf', eps=8/255, version='custom', attacks_to_run=['apgd-ce'], device=device)
aa_ce.apgd.n_restarts = 1
x_adv_ce = aa_ce.run_standard_evaluation(x_test_tensor, y_test_tensor)
aa_report_data['APGD-CE (AutoAttack)'] = evaluate_adversarial_images( x_sub, x_adv_ce, y_sub, model, 'APGD-CE')

# Run APGD-DLR
print("\nEvaluating APGD-DLR from AutoAttack...")
aa_dlr = AutoAttack(model, norm='Linf', eps=8/255, version='custom', attacks_to_run=['apgd-dlr'], device=device) 
aa_dlr.apgd.n_restarts = 1
x_adv_dlr = aa_dlr.run_standard_evaluation(x_test_tensor, y_test_tensor)
aa_report_data['APGD-DLR (AutoAttack)'] = evaluate_adversarial_images(x_sub, x_adv_dlr, y_sub, model, 'APGD-DLR')

# Run APGD-CE and APGD-DLR
print("\nEvaluating APGD-CE and APGD-DLR from AutoAttack...")
aa_both = AutoAttack(model, norm='Linf', eps=8/255, version='custom', attacks_to_run=['apgd-ce', 'apgd-dlr'], device=device)
aa_both.apgd.n_restarts = 1
x_adv_both = aa_both.run_standard_evaluation(x_test_tensor, y_test_tensor)
aa_report_data['APGD-Both (AutoAttack)'] = evaluate_adversarial_images(x_sub, x_adv_both, y_sub, model, 'APGD-Both')

# Create unified report
df_aa_methods_report = pd.DataFrame.from_dict(aa_report_data, orient='index')
df_aa_methods_report = df_aa_methods_report.round(4)

print("\n" + "="*75)
print("AUTOATTACK INDIVIDUAL METHOD STRUCTURED REPORT")
print("="*75)
print(f"Model: Carmon2019Unlabeled on CIFAR-10")
print(f"Clean accuracy: {clean_accuracy:.1%}")
print(f"AutoAttack evaluated on {len(x_sub)} test images")
print("\nComparison Table:")
display(df_aa_methods_report)
print("="*75)

Evaluating APGD-CE from AutoAttack...
using custom version including apgd-ce.
initial accuracy: 92.00%
apgd-ce - 1/1 - 19 out of 46 successfully perturbed
robust accuracy after APGD-CE: 54.00% (total time 195.3 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 54.00%

Evaluating APGD-DLR from AutoAttack...
using custom version including apgd-dlr.
initial accuracy: 92.00%
apgd-dlr - 1/1 - 19 out of 46 successfully perturbed
robust accuracy after APGD-DLR: 54.00% (total time 193.2 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 54.00%

Evaluating APGD-CE and APGD-DLR from AutoAttack...
using custom version including apgd-ce, apgd-dlr.
initial accuracy: 92.00%
apgd-ce - 1/1 - 19 out of 46 successfully perturbed
robust accuracy after APGD-CE: 54.00% (total time 180.2 s)
apgd-dlr - 1/1 - 1 out of 27 successfully perturbed
robust accuracy after APGD-DLR: 52.00% (total time 313.8 s)
max Linf pertur

,Fooling Rate (%),Mean L2 Magnitude,Mean L∞ Magnitude
APGD-CE (AutoAttack),46.0,0.6350,0.0119
APGD-DLR (AutoAttack),46.0,0.6227,0.0119
APGD-Both (AutoAttack),48.0,0.6675,0.0125


## Student Assignment: Benchmarking Alternative Robust Models

### Objective
Select an alternative robust model from the **RobustBench CIFAR-10 Linf leaderboard**, evaluate its resilience using IBM ART's standard attacks, and perform a comparative evaluation using two different, complementary AutoAttack methods.

### Tasks
1. **Model Selection:** Load a different pre-trained robust model from RobustBench (for example, `'Andriushchenko2020Understanding'` or `'Sehwag2020Hydra'`) using the `load_model` utility.
2. **ART Attack Evaluation:** Generate adversarial examples and compute the fooling rate, mean $L_2$, and mean $L_\infty$ perturbation magnitudes for:
   - **FGSM**
   - **PGD-20**
   - **C&W L2**
3. **Alternative AutoAttack Evaluation:** Standard AutoAttack includes more than just `apgd-ce` and `apgd-dlr`. Configure and run a custom `AutoAttack` pipeline using these two alternative methods:
   - **`apgd-t`** (Targeted APGD with Cross-Entropy Loss)
   - **`fab`** (Fast Adaptive Boundary Attack)
4. **Structured Report:** Assemble all results into a unified `pandas` DataFrame report comparing the clean accuracy, ART attack performance, and individual AutoAttack component stats.



# Task 1

In [43]:
# Task 1: load a different pre-trained robust model from RobustBench
model2_name = "Andriushchenko2020Understanding"
model2 = load_model(model_name=model2_name, dataset="cifar10", threat_model="Linf")
model2.eval()
model2 = model2.to(device)

classifier2 = PyTorchClassifier(
    model=model2,
    loss=nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam(model2.parameters(), lr=0.001),
    input_shape=(3, 32, 32),
    nb_classes=10,
    clip_values=(0.0, 1.0),   # keep adversarial images valid (pixels in [0, 1])
    device_type="gpu" if torch.cuda.is_available() else "cpu"
)

clean_accuracy2 = np.mean(np.argmax(classifier2.predict(x_test), axis=1) == y_test)
print(f"Model: {model2_name}")
print(f"Clean accuracy on {len(x_test)} test examples: {clean_accuracy2:.1%}")

Downloading...
From: https://drive.google.com/uc?id=1Uyvprd98bIyxfMjLdCZwm-NEJ-6GMVis
To: /Users/hamzabashar/Documents/Comp 378/Labs/Lab 3/Comp-378-labs/lesson_19/models/cifar10/Linf/Andriushchenko2020Understanding.pt
100%|██████████| 89.5M/89.5M [00:03<00:00, 23.3MB/s]


Model: Andriushchenko2020Understanding
Clean accuracy on 100 test examples: 82.0%


# Task 2


In [44]:
# Task 2: FGSM, PGD-20 and C&W L2 on the new model
art_results2 = {}

print("Running FGSM attack...")
fgsm2 = FastGradientMethod(estimator=classifier2, eps=epsilon)
x_adv_fgsm2 = fgsm2.generate(x=x_test)
evaluate_attack(classifier2, x_adv_fgsm2, x_test, y_test, 'FGSM', art_results2)

print("\nRunning PGD-20 attack...")
pgd2 = ProjectedGradientDescent(estimator=classifier2,
                                eps=epsilon,
                                eps_step=2/255,
                                max_iter=20,
                                num_random_init=1)
x_adv_pgd2 = pgd2.generate(x=x_test)
evaluate_attack(classifier2, x_adv_pgd2, x_test, y_test, 'PGD-20', art_results2)

print("\nRunning C&W L2 attack...")
cw2 = CarliniL2Method(classifier=classifier2,
                      max_iter=10,   # same as the audit above; can increase to 50
                      confidence=0.0)
x_adv_cw2 = cw2.generate(x=x_test)
evaluate_attack(classifier2, x_adv_cw2, x_test, y_test, 'C&W L2', art_results2)

Running FGSM attack...
FGSM - Fooling rate: 40.0%, Mean L2: 1.7278, Mean Linf: 0.0314

Running PGD-20 attack...


PGD-20 - Fooling rate: 44.0%, Mean L2: 1.6838, Mean Linf: 0.0314

Running C&W L2 attack...


C&W L_2: 100%|██████████| 100/100 [06:49<00:00,  4.10s/it]


C&W L2 - Fooling rate: 91.0%, Mean L2: 0.9753, Mean Linf: 0.1985


{'fooling_rate': np.float64(91.0),
 'mean_l2': np.float32(0.9753199),
 'mean_linf': np.float32(0.19845061)}

# TASK 3


In [45]:
# Task 3: custom AutoAttack pipelines with apgd-t and fab
N_AA = len(x_test)   # set to 50 if this is too slow on your computer
x_aa, y_aa = x_test[:N_AA], y_test[:N_AA]
x_aa_tensor = torch.from_numpy(x_aa).to(device)
y_aa_tensor = torch.from_numpy(y_aa).to(device)

aa_results2 = {}

# APGD-T (targeted APGD with cross-entropy loss)
print("Evaluating APGD-T from AutoAttack...")
aa_t = AutoAttack(model2, norm='Linf', eps=8/255, version='custom',
                  attacks_to_run=['apgd-t'], device=device)
aa_t.apgd_targeted.n_restarts = 1
x_adv_t = aa_t.run_standard_evaluation(x_aa_tensor, y_aa_tensor)
aa_results2['APGD-T (AutoAttack)'] = evaluate_adversarial_images(x_aa, x_adv_t, y_aa, model2, 'APGD-T')

# FAB (Fast Adaptive Boundary attack)
print("\nEvaluating FAB from AutoAttack...")
aa_fab = AutoAttack(model2, norm='Linf', eps=8/255, version='custom',
                    attacks_to_run=['fab'], device=device)
aa_fab.fab.n_restarts = 1
x_adv_fab = aa_fab.run_standard_evaluation(x_aa_tensor, y_aa_tensor)
aa_results2['FAB (AutoAttack)'] = evaluate_adversarial_images(x_aa, x_adv_fab, y_aa, model2, 'FAB')

Evaluating APGD-T from AutoAttack...
using custom version including apgd-t.
initial accuracy: 82.00%
apgd-t - 1/1 - 35 out of 82 successfully perturbed
robust accuracy after APGD-T: 47.00% (total time 362.5 s)
max Linf perturbation: 0.03137, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 47.00%

Evaluating FAB from AutoAttack...
using custom version including fab.
initial accuracy: 82.00%
fab - 1/1 - 35 out of 82 successfully perturbed
robust accuracy after FAB: 47.00% (total time 655.5 s)
max Linf perturbation: 0.02975, nan in tensor: 0, max: 1.00000, min: 0.00000
robust accuracy: 47.00%


# TASK 4

In [46]:
# Task 4: unified report for the new model
rows = {}
rows['Clean (no attack)'] = {
    'Source': '-', 'Images': len(x_test),
    'Accuracy (%)': clean_accuracy2 * 100, 'Fooling Rate (%)': (1 - clean_accuracy2) * 100,
    'Mean L2 Magnitude': 0.0, 'Mean L∞ Magnitude': 0.0,
}
for name, m in art_results2.items():
    rows[name] = {
        'Source': 'ART', 'Images': len(x_test),
        'Accuracy (%)': 100 - m['fooling_rate'], 'Fooling Rate (%)': m['fooling_rate'],
        'Mean L2 Magnitude': m['mean_l2'], 'Mean L∞ Magnitude': m['mean_linf'],
    }
for name, m in aa_results2.items():
    rows[name] = {
        'Source': 'AutoAttack', 'Images': N_AA,
        'Accuracy (%)': 100 - m['Fooling Rate (%)'], 'Fooling Rate (%)': m['Fooling Rate (%)'],
        'Mean L2 Magnitude': m['Mean L2 Magnitude'], 'Mean L∞ Magnitude': m['Mean L∞ Magnitude'],
    }

unified_report = pd.DataFrame.from_dict(rows, orient='index').round(4)

print("=" * 85)
print(f"UNIFIED ROBUSTNESS REPORT: {model2_name} on CIFAR-10 (Linf, eps = 8/255)")
print("=" * 85)
display(unified_report)

strongest = unified_report.drop(index='Clean (no attack)')['Fooling Rate (%)'].idxmax()
print(f"Strongest attack: {strongest} ({unified_report.loc[strongest, 'Fooling Rate (%)']:.1f}% fooling rate)")
print(f"Lowest accuracy under attack: {unified_report['Accuracy (%)'].min():.1f}%")
print("=" * 85)

UNIFIED ROBUSTNESS REPORT: Andriushchenko2020Understanding on CIFAR-10 (Linf, eps = 8/255)


,Source,Images,Accuracy (%),Fooling Rate (%),Mean L2 Magnitude,Mean L∞ Magnitude
Clean (no attack),-,100,82.0,18.0,0.0000,0.0000
FGSM,ART,100,60.0,40.0,1.7278,0.0314
PGD-20,ART,100,56.0,44.0,1.6838,0.0314
C&W L2,ART,100,9.0,91.0,0.9753,0.1985
APGD-T (AutoAttack),AutoAttack,100,47.0,53.0,0.5844,0.0110
FAB (AutoAttack),AutoAttack,100,47.0,53.0,0.3199,0.0059


Strongest attack: C&W L2 (91.0% fooling rate)
Lowest accuracy under attack: 9.0%
